In [1]:
# ==========================================
# KAGGLE ASL-SIGNS: TOP-200 FINE-TUNING
# ==========================================
# This notebook fine-tunes the Kaggle 1st-place ASL model on top-200 conversational signs

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

import numpy as np
import pandas as pd
import json
import glob
from tqdm import tqdm
import collections
import gc
import math

print("✅ Imports loaded successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

E0000 00:00:1764175631.310035      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764175631.359921      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

✅ Imports loaded successfully!
TensorFlow version: 2.18.0
GPU Available: True


In [2]:
# ==========================================
# 1. PATHS & CONFIGURATION
# ==========================================

# Kaggle ASL-Signs Dataset (landmarks already extracted!)
TRAIN_CSV = "/kaggle/input/asl-signs/train.csv"
SIGN_MAP_JSON = "/kaggle/input/asl-signs/sign_to_prediction_index_map.json"
LANDMARK_DIR = "/kaggle/input/asl-signs/train_landmark_files"

# Pretrained Kaggle 1st-place weights (250 ASL signs)
WEIGHTS_PATH = "/kaggle/input/islr-weights/islr-fp16-192-8-seed45-foldall-last.h5"

# Model constants
ROWS_PER_FRAME = 543  # MediaPipe landmarks: face(468) + left_hand(21) + pose(33) + right_hand(21)
MAX_LEN = 128  # Sequence length (shorter than 384 for faster training)
BATCH_SIZE = 32

# Training config
TOP_K = 200  # Select top-200 most conversational signs
MIN_SAMPLES_PER_SIGN = 50  # Minimum training samples per sign

print(f"✅ Configuration loaded")
print(f"   - Landmark directory: {LANDMARK_DIR}")
print(f"   - Pretrained weights: {WEIGHTS_PATH}")
print(f"   - Target vocab size: {TOP_K} signs")
print(f"   - Sequence length: {MAX_LEN} frames")

✅ Configuration loaded
   - Landmark directory: /kaggle/input/asl-signs/train_landmark_files
   - Pretrained weights: /kaggle/input/islr-weights/islr-fp16-192-8-seed45-foldall-last.h5
   - Target vocab size: 200 signs
   - Sequence length: 128 frames


In [3]:
# ==========================================
# 2. LOAD KAGGLE ASL-SIGNS METADATA
# ==========================================

# Load train.csv (contains participant_id, path, sign labels)
train_df = pd.read_csv(TRAIN_CSV)
print(f"✅ Loaded {len(train_df)} training samples")
print(f"   Columns: {list(train_df.columns)}")
print(f"\nFirst 3 rows:")
print(train_df.head(3))

# Load sign-to-index mapping
with open(SIGN_MAP_JSON, 'r') as f:
    sign_to_idx = json.load(f)
    
idx_to_sign = {v: k for k, v in sign_to_idx.items()}
print(f"\n✅ Loaded {len(sign_to_idx)} sign labels")
print(f"   Example signs: {list(sign_to_idx.keys())[:10]}")

✅ Loaded 94477 training samples
   Columns: ['path', 'participant_id', 'sequence_id', 'sign']

First 3 rows:
                                            path  participant_id  sequence_id  \
0  train_landmark_files/26734/1000035562.parquet           26734   1000035562   
1  train_landmark_files/28656/1000106739.parquet           28656   1000106739   
2   train_landmark_files/16069/100015657.parquet           16069    100015657   

    sign  
0   blow  
1   wait  
2  cloud  

✅ Loaded 250 sign labels
   Example signs: ['TV', 'after', 'airplane', 'all', 'alligator', 'animal', 'another', 'any', 'apple', 'arm']


In [4]:
# ==========================================
# 3. SELECT TOP-200 CONVERSATIONAL SIGNS
# ==========================================

# Count samples per sign
sign_counts = train_df['sign'].value_counts()
print(f"Total unique signs in dataset: {len(sign_counts)}")
print(f"\nTop 10 most frequent signs:")
print(sign_counts.head(10))

# Filter: only signs with >= MIN_SAMPLES_PER_SIGN
valid_signs = sign_counts[sign_counts >= MIN_SAMPLES_PER_SIGN]
print(f"\n✅ Signs with ≥{MIN_SAMPLES_PER_SIGN} samples: {len(valid_signs)}")

# Select top-K
top_k_signs = valid_signs.head(TOP_K).index.tolist()
print(f"✅ Selected top-{TOP_K} signs")
print(f"\nTop-20 selected signs:")
for i, sign in enumerate(top_k_signs[:20], 1):
    count = sign_counts[sign]
    print(f"  {i:2d}. {sign:20s} ({count:4d} samples)")

# Create new label mapping (0 to TOP_K-1)
new_sign_to_idx = {sign: i for i, sign in enumerate(top_k_signs)}
new_idx_to_sign = {i: sign for sign, i in new_sign_to_idx.items()}

# Filter train_df to only include top-K signs
train_df_filtered = train_df[train_df['sign'].isin(top_k_signs)].copy()
train_df_filtered['label'] = train_df_filtered['sign'].map(new_sign_to_idx)

print(f"\n✅ Filtered dataset: {len(train_df_filtered)} samples across {TOP_K} signs")
print(f"   Average samples per sign: {len(train_df_filtered) / TOP_K:.1f}")

Total unique signs in dataset: 250

Top 10 most frequent signs:
sign
listen     415
look       414
shhh       411
donkey     410
mouse      408
hear       405
uncle      405
duck       405
bird       404
pretend    404
Name: count, dtype: int64

✅ Signs with ≥50 samples: 250
✅ Selected top-200 signs

Top-20 selected signs:
   1. listen               ( 415 samples)
   2. look                 ( 414 samples)
   3. shhh                 ( 411 samples)
   4. donkey               ( 410 samples)
   5. mouse                ( 408 samples)
   6. hear                 ( 405 samples)
   7. uncle                ( 405 samples)
   8. duck                 ( 405 samples)
   9. bird                 ( 404 samples)
  10. pretend              ( 404 samples)
  11. cow                  ( 404 samples)
  12. brown                ( 403 samples)
  13. who                  ( 403 samples)
  14. sleepy               ( 403 samples)
  15. bye                  ( 402 samples)
  16. nuts                 ( 402 samples)
  1

In [5]:
# ==========================================
# 4. TRAIN/VAL SPLIT (PARTICIPANT-BASED)
# ==========================================
# Split by participant to ensure no data leakage

from sklearn.model_selection import train_test_split

# Get unique participants
participants = train_df_filtered['participant_id'].unique()
print(f"Total participants: {len(participants)}")

# Split participants 80/20
train_participants, val_participants = train_test_split(
    participants, test_size=0.2, random_state=42
)

# Split data based on participants
train_data = train_df_filtered[train_df_filtered['participant_id'].isin(train_participants)]
val_data = train_df_filtered[train_df_filtered['participant_id'].isin(val_participants)]

print(f"\n✅ Train/Val split:")
print(f"   Train: {len(train_data)} samples ({len(train_participants)} participants)")
print(f"   Val:   {len(val_data)} samples ({len(val_participants)} participants)")
print(f"\n   Train signs: {train_data['sign'].nunique()}")
print(f"   Val signs:   {val_data['sign'].nunique()}")

Total participants: 21

✅ Train/Val split:
   Train: 58004 samples (16 participants)
   Val:   19076 samples (5 participants)

   Train signs: 200
   Val signs:   200


In [6]:
# ==========================================
# 5. LANDMARK LOADING & PREPROCESSING
# ==========================================

def load_landmark_parquet(participant_id, sequence_id):
    """Load landmark parquet file for a given sample."""
    path = f"{LANDMARK_DIR}/{participant_id}/{sequence_id}.parquet"
    try:
        df = pd.read_parquet(path)
        return df
    except:
        return None

def preprocess_landmarks(df):
    """Convert landmark dataframe to (T, 543, 3) array."""
    if df is None or len(df) == 0:
        return None
    
    n_frames = df['frame'].max() + 1
    
    # Initialize array
    data = np.zeros((n_frames, ROWS_PER_FRAME, 3), dtype=np.float32)
    
    # Fill in landmarks
    for _, row in df.iterrows():
        frame_idx = int(row['frame'])
        lm_idx = int(row['landmark_index'])
        
        # Map type to global index
        if row['type'] == 'face':
            global_idx = lm_idx
        elif row['type'] == 'left_hand':
            global_idx = 468 + lm_idx
        elif row['type'] == 'pose':
            global_idx = 489 + lm_idx
        elif row['type'] == 'right_hand':
            global_idx = 522 + lm_idx
        else:
            continue
        
        if global_idx < ROWS_PER_FRAME and frame_idx < n_frames:
            data[frame_idx, global_idx, 0] = row['x']
            data[frame_idx, global_idx, 1] = row['y']
            data[frame_idx, global_idx, 2] = row['z']
    
    return data

def resize_pad_sequence(data, target_len=MAX_LEN):
    """Resize or pad sequence to target length."""
    current_len = data.shape[0]
    
    if current_len == target_len:
        return data
    elif current_len < target_len:
        # Pad with zeros
        pad_len = target_len - current_len
        padding = np.zeros((pad_len, ROWS_PER_FRAME, 3), dtype=np.float32)
        return np.concatenate([data, padding], axis=0)
    else:
        # Downsample using linear interpolation
        indices = np.linspace(0, current_len - 1, target_len)
        resampled = np.zeros((target_len, ROWS_PER_FRAME, 3), dtype=np.float32)
        for i, idx in enumerate(indices):
            idx_low = int(np.floor(idx))
            idx_high = min(int(np.ceil(idx)), current_len - 1)
            weight = idx - idx_low
            resampled[i] = (1 - weight) * data[idx_low] + weight * data[idx_high]
        return resampled

print("✅ Landmark loading functions defined")

✅ Landmark loading functions defined


In [ ]:
# ==========================================
# 6. CREATE TFRECORDS (PARALLELIZED WITH STREAMING)
# ==========================================
import multiprocessing as mp
from functools import partial
import pickle
import tempfile
import shutil

print("Creating TFRecord files (using 4 CPU cores)...\n")

# Serialization functions
def serialize_example(landmarks, label):
    """Serialize landmarks and label to TFRecord format."""
    # Flatten and convert to float16
    landmarks_flat = landmarks.reshape(-1).astype(np.float16)
    
    feature = {
        'video': tf.train.Feature(bytes_list=tf.train.BytesList(value=[landmarks_flat.tobytes()])),
        'label': tf.train.Feature(int64_list=tf.train.Int64List(value=[label])),
    }
    
    example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
    return example_proto.SerializeToString()

def load_landmarks(participant_id, sequence_id):
    """Load and preprocess landmarks for a sample."""
    # Load parquet
    df = load_landmark_parquet(participant_id, sequence_id)
    if df is None:
        return None
    
    # Preprocess
    landmarks = preprocess_landmarks(df)
    if landmarks is None:
        return None
    
    # Resize/pad to MAX_LEN
    landmarks = resize_pad_sequence(landmarks, MAX_LEN)
    return landmarks

def process_single_sample(row_tuple):
    """Process a single sample (for parallel execution)."""
    idx, row = row_tuple
    try:
        # Load landmarks
        landmarks = load_landmarks(row['participant_id'], row['sequence_id'])
        if landmarks is None:
            return idx, None
        
        # Get label
        label = row['label']
        
        # Serialize
        example = serialize_example(landmarks, label)
        return idx, example
    except Exception as e:
        return idx, None

def create_tfrecord_parallel_streaming(df, output_path, num_workers=4, resume=True):
    """Create TFRecord file using parallel processing with streaming writes."""
    
    checkpoint_path = output_path.replace('.tfrecord.gz', '_checkpoint.pkl')
    temp_path = output_path + '.tmp'
    
    # Check if we should resume
    processed_indices = set()
    if resume and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                processed_indices = pickle.load(f)
            print(f"📂 Found checkpoint: {len(processed_indices)} samples already processed")
            
            # Copy existing partial file to temp
            if os.path.exists(output_path):
                shutil.copy(output_path, temp_path)
        except:
            processed_indices = set()
    
    # Filter out already processed samples
    df_reset = df.reset_index(drop=True)
    remaining_rows = [(idx, row) for idx, row in df_reset.iterrows() if idx not in processed_indices]
    
    if len(remaining_rows) == 0:
        print(f"✅ All samples already processed for {output_path.split('/')[-1]}")
        return
    
    print(f"Processing {len(remaining_rows)} remaining samples (out of {len(df)} total)...")
    
    # Open TFRecord writer
    writer = tf.io.TFRecordWriter(temp_path, options='GZIP')
    
    try:
        # Process in parallel with streaming writes and smooth progress
        samples_written = 0
        
        # Create manual progress bar for smooth updates
        pbar = tqdm(total=len(remaining_rows), desc=f"Processing {output_path.split('/')[-1]}")
        
        with mp.Pool(num_workers) as pool:
            # Use imap_unordered with chunksize for speed
            for idx, example in pool.imap_unordered(process_single_sample, remaining_rows, chunksize=50):
                if example is not None:
                    # Write immediately to disk
                    writer.write(example)
                    processed_indices.add(idx)
                    samples_written += 1
                    
                    # Update progress bar smoothly (every sample!)
                    pbar.update(1)
                    
                    # Save checkpoint every 1000 samples
                    if samples_written % 1000 == 0:
                        writer.flush()  # Ensure data is written to disk
                        with open(checkpoint_path, 'wb') as f:
                            pickle.dump(processed_indices, f)
                        
                        # Show file size progress
                        if os.path.exists(temp_path):
                            size_mb = os.path.getsize(temp_path) / (1024 * 1024)
                            pbar.write(f"   💾 Checkpoint: {samples_written} samples, {size_mb:.1f} MB")
        
        pbar.close()
    
    finally:
        # Close writer
        writer.close()
    
    # Move temp file to final location
    if os.path.exists(temp_path):
        shutil.move(temp_path, output_path)
    
    # Final file size
    final_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"✅ Created {output_path}")
    print(f"   Samples: {len(processed_indices)}")
    print(f"   Size: {final_size_mb:.1f} MB\n")
    
    # Clean up checkpoint
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

# Create TFRecords using all 4 cores with streaming writes
TRAIN_TFRECORD = "/kaggle/working/train_top200.tfrecord.gz"
VAL_TFRECORD = "/kaggle/working/val_top200.tfrecord.gz"

# Store sample counts for later use
NUM_TRAIN = len(train_data)
NUM_VAL = len(val_data)

create_tfrecord_parallel_streaming(train_data, TRAIN_TFRECORD, num_workers=4, resume=True)
create_tfrecord_parallel_streaming(val_data, VAL_TFRECORD, num_workers=4, resume=True)

print(f"✅ TFRecord creation complete!")
print(f"   Train: {TRAIN_TFRECORD} ({NUM_TRAIN} samples)")
print(f"   Val:   {VAL_TFRECORD} ({NUM_VAL} samples)")

Creating TFRecord files (using 4 CPU cores)...

📂 Found checkpoint: 2000 samples already processed
Processing 56004 remaining samples (out of 58004 total)...


Processing train_top200.tfrecord.gz:   2%|▏         | 1000/56004 [06:38<2:05:32,  7.30it/s]

   💾 Checkpoint: 1000 samples, 73.2 MB


Processing train_top200.tfrecord.gz:   4%|▎         | 2000/56004 [13:03<52:07, 17.26it/s]   

   💾 Checkpoint: 2000 samples, 149.2 MB


...

Processing train_top200.tfrecord.gz:  89%|████████▉ | 50000/56004 [5:07:47<14:32,  6.88it/s]  

   💾 Checkpoint: 50000 samples, 3685.5 MB


Processing train_top200.tfrecord.gz:  91%|█████████ | 51000/56004 [5:14:05<04:39, 17.89it/s]  

   💾 Checkpoint: 51000 samples, 3760.5 MB


Processing train_top200.tfrecord.gz:  93%|█████████▎| 51974/56004 [5:20:30<32:48,  2.05it/s]  